# Synthetic portfolio factor regressions

Monthly USD total returns use the synthetic-sensitivity portfolio from `main.ipynb`. Each model uses Newey–West/HAC standard errors with three lags. Coefficients describe realized co-movement, not causal or investable factor replication.


In [1]:
from pathlib import Path
from io import BytesIO, StringIO
from urllib.request import urlopen
from zipfile import ZipFile

import pandas as pd
import statsmodels.api as sm
from IPython.display import display

from helpers.backtest import align_prices, load_ntsg_usd, load_prices, monthly_rebalanced_returns, splice_proxy_to_actual

MODEL_WEIGHTS = pd.Series({"NTSG": 0.60, "COM": 0.10, "DBMF": 0.15, "GDE": 0.10, "TAIL": 0.05})
BENCHMARK_WEIGHTS = pd.Series({"VT": 0.60, "BNDW": 0.40})
HAC_LAGS = 3


def monthly_returns(daily_returns: pd.Series) -> pd.Series:
    """Compound daily returns, excluding the incomplete first calendar month."""
    complete_returns = daily_returns.iloc[1:]
    returns = (1 + complete_returns).resample("ME").prod() - 1
    return returns[returns.index.to_period("M") > complete_returns.index[0].to_period("M")]


def regression_coefficients(excess_returns: pd.DataFrame, factors: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Fit one HAC OLS regression per strategy and return coefficient and fit tables."""
    data = pd.concat([excess_returns, factors], axis=1).dropna()
    coefficient_tables = []
    fit_rows = []
    for strategy in excess_returns:
        model = sm.OLS(data[strategy], sm.add_constant(data[factors.columns])).fit(
            cov_type="HAC", cov_kwds={"maxlags": HAC_LAGS}
        )
        coefficient_tables.append(pd.DataFrame({
            "coefficient": model.params,
            "hac_t_stat": model.tvalues,
            "hac_p_value": model.pvalues,
            "ci_lower": model.conf_int().iloc[:, 0],
            "ci_upper": model.conf_int().iloc[:, 1],
        }).rename_axis("term").assign(strategy=strategy).set_index("strategy", append=True))
        fit_rows.append({"strategy": strategy, "observations": int(model.nobs), "r_squared": model.rsquared})
    coefficients = pd.concat(coefficient_tables).reorder_levels([1, 0]).sort_index()
    return coefficients, pd.DataFrame(fit_rows).set_index("strategy")


# Rebuild the main notebook's synthetic-sensitivity portfolio with its original proxy/actual splices.
extended_prices = pd.concat([
    splice_proxy_to_actual("NTSG_proxy_extended.csv", load_ntsg_usd()),
    load_prices("COM.CSV", "COM"),
    load_prices("DBMF.CSV", "DBMF"),
    splice_proxy_to_actual("GDE_proxy.csv", load_prices("GDE.CSV", "GDE")),
    load_prices("TAIL.CSV", "TAIL"),
], axis=1, sort=True)
benchmark_prices = pd.concat([load_prices("VT.CSV", "VT"), load_prices("BNDW.CSV", "BNDW")], axis=1, sort=True)
prices = align_prices(pd.concat([extended_prices, benchmark_prices], axis=1, sort=True))

strategy_returns = pd.DataFrame({
    "Model portfolio": monthly_returns(monthly_rebalanced_returns(prices, MODEL_WEIGHTS)),
    "60/40 VT/BNDW": monthly_returns(monthly_rebalanced_returns(prices, BENCHMARK_WEIGHTS)),
    "VT": monthly_returns(prices["VT"].pct_change().fillna(0.0)),
})
cash_returns = monthly_returns(load_prices("USD_CASH_EFFR.CSV", "USD cash").pct_change().fillna(0.0))
print(f"Strategy sample: {strategy_returns.index.min():%Y-%m} to {strategy_returns.index.max():%Y-%m}")


Strategy sample: 2019-06 to 2026-09


## Economic exposure model

For each strategy, monthly USD excess return is regressed on the excess returns of VT (global equities), BNDW (global bonds), PDBC (commodities), WTMF (managed futures), and GLD (gold). Factors are ETF total returns, so their embedded costs are retained.

$$r_{p,t}-r_{f,t}=\alpha+\beta_E r_{VT,t}^{e}+\beta_B r_{BNDW,t}^{e}+\beta_C r_{PDBC,t}^{e}+\beta_T r_{WTMF,t}^{e}+\beta_G r_{GLD,t}^{e}+\epsilon_t$$


In [2]:
# Dedicated run: economic exposure model.
economic_prices = pd.concat([
    load_prices("VT.CSV", "VT excess"),
    load_prices("BNDW.CSV", "BNDW excess"),
    load_prices("PDBC.CSV", "PDBC excess"),
    load_prices("WTMF.CSV", "WTMF excess"),
    load_prices("GLD.CSV", "GLD excess"),
], axis=1, sort=True)
economic_returns = economic_prices.resample("ME").last().pct_change()
economic_factors = economic_returns.sub(cash_returns, axis=0)
economic_coefficients, economic_fit = regression_coefficients(
    strategy_returns.sub(cash_returns, axis=0), economic_factors
)
print("Economic exposure model coefficients (monthly decimal returns)")
display(economic_coefficients.round(4))
display(economic_fit.round(4))


Economic exposure model coefficients (monthly decimal returns)


coefficient    hac_t_stat  hac_p_value  ci_lower  \
strategy        term                                                            
60/40 VT/BNDW   BNDW excess       0.4000  1.412272e+13       0.0000    0.4000   
                GLD excess        0.0000  2.240000e-02       0.9821   -0.0000   
                PDBC excess      -0.0000 -1.068000e-01       0.9150   -0.0000   
                VT excess         0.6000  6.334284e+13       0.0000    0.6000   
                WTMF excess       0.0000  7.070000e-02       0.9436   -0.0000   
                const             0.0000  4.598100e+00       0.0000    0.0000   
Model portfolio BNDW excess       0.3085  3.873500e+00       0.0001    0.1524   
                GLD excess        0.1182  6.591400e+00       0.0000    0.0831   
                PDBC excess       0.0130  6.818000e-01       0.4954   -0.0243   
                VT excess         0.5964  3.032050e+01       0.0000    0.5578   
                WTMF excess       0.1518  3.764500e+00       0.0002    0.0728   
                const             0.0006  7.239000e-01       0.4691   -0.0009   
VT              BNDW excess      -0.0000 -1.244000e-01       0.9010   -0.0000   
                GLD excess        0.0000  7.500000e-03       0.9940   -0.0000   
                PDBC excess      -0.0000 -3.220000e-02       0.9743   -0.0000   
                VT excess         1.0000  7.231284e+13       0.0000    1.0000   
                WTMF excess       0.0000  6.890000e-02       0.9451   -0.0000   
                const             0.0000  4.568300e+00       0.0000    0.0000   

                             ci_upper  
strategy        term                   
60/40 VT/BNDW   BNDW excess    0.4000  
                GLD excess     0.0000  
                PDBC excess    0.0000  
                VT excess      0.6000  
                WTMF excess    0.0000  
                const          0.0000  
Model portfolio BNDW excess    0.4647  
                GLD excess     0.1534  
                PDBC excess    0.0503  
                VT excess      0.6349  
                WTMF excess    0.2309  
                const          0.0021  
VT              BNDW excess    0.0000  
                GLD excess     0.0000  
                PDBC excess    0.0000  
                VT excess      1.0000  
                WTMF excess    0.0000  
                const          0.0000

,observations,r_squared
strategy,,
Model portfolio,88,0.9598
60/40 VT/BNDW,88,1.0000
VT,88,1.0000


## Fama–French five-factor model plus momentum

The official Ken French monthly US factors are downloaded at execution. The dependent variable is each strategy's monthly USD total return less the Fama–French risk-free rate. This is an equity-style attribution: it does not fully represent the portfolio's global bond, commodity, managed-futures, gold, or tail-option sleeves.

$$r_{p,t}-r_{f,t}=\alpha+\beta_M MKT_t+\beta_S SMB_t+\beta_V HML_t+\beta_P RMW_t+\beta_I CMA_t+\beta_U UMD_t+\epsilon_t$$


In [3]:
# Dedicated run: Fama–French five factors plus momentum.
FF5_URL = "https://mba.tuck.dartmouth.edu/pages/faculty/ken.french/ftp/F-F_Research_Data_5_Factors_2x3_CSV.zip"
MOMENTUM_URL = "https://mba.tuck.dartmouth.edu/pages/faculty/ken.french/ftp/F-F_Momentum_Factor_CSV.zip"


def french_monthly_factors(url: str, required_columns: list[str]) -> pd.DataFrame:
    """Download and parse the monthly section of an official Ken French CSV archive."""
    with ZipFile(BytesIO(urlopen(url).read())) as archive:
        csv_name = next(name for name in archive.namelist() if name.lower().endswith(".csv"))
        lines = archive.read(csv_name).decode("latin-1").splitlines()
    header = next(
        index
        for index, line in enumerate(lines)
        if line.lstrip().startswith(",") and all(column in line for column in required_columns)
    )
    end = next(index for index in range(header + 1, len(lines)) if not lines[index].strip())
    table = pd.read_csv(StringIO("\n".join(lines[header:end])))
    date_column = table.columns[0]
    table[date_column] = table[date_column].astype(str).str.strip()
    table = table[table[date_column].str.fullmatch(r"\d{6}")].copy()
    table.index = pd.to_datetime(table.pop(date_column), format="%Y%m") + pd.offsets.MonthEnd(0)
    return table[required_columns].apply(pd.to_numeric, errors="raise").div(100)


ff5 = french_monthly_factors(FF5_URL, ["Mkt-RF", "SMB", "HML", "RMW", "CMA", "RF"])
momentum = french_monthly_factors(MOMENTUM_URL, ["Mom"]).rename(columns={"Mom": "UMD"})
ff_factors = ff5.join(momentum, how="inner")
factor_columns = ["Mkt-RF", "SMB", "HML", "RMW", "CMA", "UMD"]
ff_coefficients, ff_fit = regression_coefficients(
    strategy_returns.sub(ff_factors["RF"], axis=0), ff_factors[factor_columns]
)
print(f"Fama–French sample: {ff_factors.index.min():%Y-%m} to {ff_factors.index.max():%Y-%m}")
print("Fama–French five-factor plus momentum coefficients (monthly decimal returns)")
display(ff_coefficients.round(4))
display(ff_fit.round(4))


Fama–French sample: 1963-07 to 2026-09
Fama–French five-factor plus momentum coefficients (monthly decimal returns)


coefficient  hac_t_stat  hac_p_value  ci_lower  \
strategy        term                                                     
60/40 VT/BNDW   CMA          0.0509      0.9088       0.3634   -0.0589   
                HML          0.0150      0.4222       0.6729   -0.0545   
                Mkt-RF       0.6109     30.4813       0.0000    0.5716   
                RMW         -0.0026     -0.0692       0.9448   -0.0753   
                SMB         -0.0055     -0.1376       0.8905   -0.0837   
                UMD         -0.0402     -1.3385       0.1807   -0.0990   
                const       -0.0018     -1.9638       0.0496   -0.0037   
Model portfolio CMA          0.0882      1.1017       0.2706   -0.0687   
                HML         -0.0144     -0.2861       0.7748   -0.1129   
                Mkt-RF       0.6652     26.9669       0.0000    0.6168   
                RMW          0.0641      1.2086       0.2268   -0.0398   
                SMB         -0.0528     -1.0255       0.3051   -0.1537   
                UMD         -0.0189     -0.4871       0.6262   -0.0952   
                const       -0.0002     -0.1067       0.9151   -0.0029   
VT              CMA          0.0879      1.5611       0.1185   -0.0225   
                HML          0.0615      1.5093       0.1312   -0.0184   
                Mkt-RF       0.9137     48.6378       0.0000    0.8768   
                RMW         -0.0116     -0.3161       0.7520   -0.0832   
                SMB         -0.0125     -0.3223       0.7472   -0.0886   
                UMD         -0.0493     -1.5623       0.1182   -0.1111   
                const       -0.0010     -0.9517       0.3412   -0.0029   

                        ci_upper  
strategy        term              
60/40 VT/BNDW   CMA       0.1606  
                HML       0.0844  
                Mkt-RF    0.6502  
                RMW       0.0701  
                SMB       0.0728  
                UMD       0.0187  
                const    -0.0000  
Model portfolio CMA       0.2450  
                HML       0.0841  
                Mkt-RF    0.7135  
                RMW       0.1680  
                SMB       0.0481  
                UMD       0.0573  
                const     0.0026  
VT              CMA       0.1984  
                HML       0.1413  
                Mkt-RF    0.9505  
                RMW       0.0601  
                SMB       0.0636  
                UMD       0.0125  
                const     0.0010

,observations,r_squared
strategy,,
Model portfolio,88,0.8629
60/40 VT/BNDW,88,0.9214
VT,88,0.9588
